# DATASET PLN

En este cuadernillo se recoge el proceso de contrucción del dataset de PLN. Este dataset se utiliza en filtrado basado en contenido

In [ ]:
import pandas as pd

## Agregación de tags.parquet
El archivo tags_integrity.parquet contiene una fila por cada etiqueta que los usuarios han valorado, es decir, una película puede aparecer en múltiples filas. Mediante este proceso de agregación, se combinan todas las etiquetas asociadas por películas, de forma que cada fila almacena todas las etiquetas que los usuarios han dado a esa película.

In [2]:
tags = pd.read_parquet('../data/02_processed/tags_integrity.parquet')
listaTags = tags.groupby('movieId')['tag'].agg(list).reset_index()
listaTags

,movieId,tag
0,1,"[pixar, pixar, fun]"
1,2,"[fantasy, magic board game, robin williams, game]"
2,3,"[moldy, old]"
3,5,"[pregnancy, remake]"
4,7,[remake]
...,...,...
1567,183611,"[comedy, funny, rachel mcadams]"
1568,184471,"[adventure, alicia vikander, video game adapta..."
1569,187593,"[josh brolin, ryan reynolds, sarcasm]"
1570,187595,"[emilia clarke, star wars]"


In [3]:
listaTags['tag'] = listaTags['tag'].str.join(" ")
listaTags.head()


,movieId,tag
0,1,pixar pixar fun
1,2,fantasy magic board game robin williams game
2,3,moldy old
3,5,pregnancy remake
4,7,remake


## Combinación de TMDB.JSON 

## Unificación de datos para PLN
Se combina la información para crear un conjunto de datos  que sirva para aplicar técnicas de procesamiento del lenguaje ntural.

Se crea un datset con la siguiente información.
- movieId (movies_integrity.parquet)
- title (movies_integrity.parquet)
- tags (tags_integrity.parquet)
- genres (movies_integrity.parquet)
- overview  (TMDB_clean.parquet)

Se usa  como tabla base movies_integrity.parquet y se hacen left joins con los demás archivos

In [16]:
TMDB = pd.read_parquet('../data/02_processed/TMDB_clean.parquet')
TMDB = TMDB.rename(columns={'id': 'tmdbId'})  
links = pd.read_parquet('../data/02_processed/links_integrity.parquet')

combinacion = pd.merge(TMDB, links, how='inner', on='tmdbId')
combinacion.info()


<class 'pandas.DataFrame'>
RangeIndex: 9616 entries, 0 to 9615
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   tmdbId        9616 non-null   int64  
 1   title         9616 non-null   str    
 2   genres        9616 non-null   object 
 3   popularity    9616 non-null   float64
 4   overview      9616 non-null   str    
 5   tagline       9616 non-null   str    
 6   vote_average  9616 non-null   float64
 7   vote_count    9616 non-null   int64  
 8   runtime       9616 non-null   int64  
 9   budget        9616 non-null   int64  
 10  revenue       9616 non-null   int64  
 11  movieId       9616 non-null   int64  
dtypes: float64(2), int64(6), object(1), str(3)
memory usage: 4.0+ MB


In [ ]:
movies = pd.read_parquet('../data/02_processed/movies_integrity.parquet')
movies = movies[['movieId', 'title', 'genres']]

movieInfo = combinacion[['movieId', 'overview']]

In [7]:
tabla1 = pd.merge(movies, listaTags, how = "left", on = "movieId")
tabla2 = pd.merge(tabla1, movieInfo,how = "left", on = "movieId" )

La columna sinopsis se unifica el texto, eliminando carácteres especiales, pasando a minúscula y conservando espacios.  
La columna de géneros se cambia | por un espacio y se escriben en minúsculas. 

In [ ]:
tabla2['genres'] = tabla2['genres'].str.replace("|", " ").str.lower()
tabla2['genres'] = tabla2['genres'].str.replace('sci-fi', 'scifi', case=False)
tabla2['genres'] = tabla2['genres'].str.replace('film-noir', 'filmnoir', case=False)      
tabla2['tag'] = tabla2['tag'].str.replace(r'[^a-zA-Z\s]', '', regex=True).str.lower()       

In [9]:
tabla2.info()


<class 'pandas.DataFrame'>
RangeIndex: 9737 entries, 0 to 9736
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   movieId   9737 non-null   int64 
 1   title     9737 non-null   str   
 2   genres    9703 non-null   str   
 3   tag       1572 non-null   object
 4   overview  9616 non-null   str   
dtypes: int64(1), object(1), str(3)
memory usage: 3.3+ MB


In [10]:
tabla2.head()

,movieId,title,genres,tag,overview
0,1,Toy Story,adventure animation children comedy fantasy,pixar pixar fun,"Led by Woody, Andy's toys live happily in his ..."
1,2,Jumanji,adventure children fantasy,fantasy magic board game robin williams game,When siblings Judy and Peter discover an encha...
2,3,Grumpier Old Men,comedy romance,moldy old,A family wedding reignites the ancient feud be...
3,4,Waiting to Exhale,comedy drama romance,NaN,"Cheated on, mistreated and stepped on, the wom..."
4,5,Father of the Bride Part II,comedy,pregnancy remake,Just when George Banks has recovered from his ...


Se rellenan espacios NAN con " " para facilitar la aplicacion de herramientas de PLN

In [11]:
tabla2['genres'] = tabla2['genres'].fillna("")
tabla2['tag'] = tabla2['tag'].fillna("")
tabla2['overview'] = tabla2['overview'].fillna("")
tabla2['join'] =tabla2['genres'] + " " +tabla2['tag'] + " " + tabla2['overview']

In [12]:
tabla2.info()

<class 'pandas.DataFrame'>
RangeIndex: 9737 entries, 0 to 9736
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   movieId   9737 non-null   int64 
 1   title     9737 non-null   str   
 2   genres    9737 non-null   str   
 3   tag       9737 non-null   object
 4   overview  9737 non-null   str   
 5   join      9737 non-null   str   
dtypes: int64(1), object(1), str(4)
memory usage: 6.1+ MB


In [13]:
tabla2.head()

,movieId,title,genres,tag,overview,join
0,1,Toy Story,adventure animation children comedy fantasy,pixar pixar fun,"Led by Woody, Andy's toys live happily in his ...",adventure animation children comedy fantasy pi...
1,2,Jumanji,adventure children fantasy,fantasy magic board game robin williams game,When siblings Judy and Peter discover an encha...,adventure children fantasy fantasy magic board...
2,3,Grumpier Old Men,comedy romance,moldy old,A family wedding reignites the ancient feud be...,comedy romance moldy old A family wedding reig...
3,4,Waiting to Exhale,comedy drama romance,,"Cheated on, mistreated and stepped on, the wom...","comedy drama romance Cheated on, mistreated a..."
4,5,Father of the Bride Part II,comedy,pregnancy remake,Just when George Banks has recovered from his ...,comedy pregnancy remake Just when George Banks...


In [15]:
tabla2.to_parquet('../data/03_model_ready/NLP.parquet')